In [2]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_feather("../Outputs/26_df.feather")

In [5]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'rent_mode',
       'rent_value', 'rent_to_single', 'rent_type', 'price_mode',
       'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform',
       'transformable_price', 'transformable_credit', 'transformed_credit',
       'transformable_rent', 'transformed_rent', 'land_size', 'building_size',
       'deed_type', 'has_business_deed', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas',
       'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'regular_person_capacity', 'extra_person

In [6]:
df["building_age"] = 1404 - df["construction_year"]

In [7]:
df["age_group"] = pd.cut(
    df["building_age"],
    bins=[-1, 5, 10, 20, 30, float("inf")],
    labels= ["نوساز", "کم سن", "میان سال","قدیمی", "خیلی قدیمی"]
)

In [26]:
df["area_group"] = pd.cut(
    df["building_size"],
    bins=[0, 50, 100, 150, 200, float("inf")],
    labels=["کوچک", "متوسط", "بزرگ", "خیلی بزرگ", "بسیار بزرگ"]
)

In [9]:
amenity_columns = [
    "has_balcony",
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_water",
    "has_warm_water_provider",
    "has_electricity",
    "has_gas",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom"
]

df["amenity_count"] = df[amenity_columns].eq(True).sum(axis=1)

In [10]:
df["amenity_level"] = pd.cut(
    df["amenity_count"],
    bins=[-1, 2, 6, 11, float("inf")],
    labels=["کم", "متوسط", "زیاد", "لوکس"]
)

In [11]:
luxury_amenity_columns = [  
"has_security_guard",
    "has_barbecue",
    "has_pool",
    "has_jacuzzi",
    "has_sauna"]
df["luxury_amenity_count"] = df[luxury_amenity_columns].eq(True).sum(axis=1)

In [12]:
df["luxury_amenity_level"] = pd.cut(
    df["luxury_amenity_count"],
    bins=[-1, 2, 4, 5, float("inf")],
    labels=["کم", "متوسط", "زیاد", "لوکس"])

In [13]:
df["city_neighborhood"] = (
    df["city_slug"].astype(str) + "|" +
    df["neighborhood_slug"].astype(str)
)

In [14]:
df["full_property_type"] = (df["cat2_slug"].astype(str) + "|" + df["cat3_slug"].astype(str))

In [15]:
df["month_number"] = df["created_at_month"].dt.month

In [16]:
df["month_name"] = df["created_at_month"].dt.month_name()

In [17]:
df["season"] = df["month_number"].map({
    1: "زمستان",
    2: "زمستان",
    3: "بهار",
    4: "بهار",
    5: "بهار",
    6: "تابستان",
    7: "تابستان",
    8: "تابستان",
    9: "پاییز",
    10: "پاییز",
    11: "پاییز",
    12: "زمستان"
})

In [18]:
feature_columns = [
    "building_age",
    "age_group",
    "area_group",
    "amenity_count",
    "amenity_level",
    "luxury_amenity_count",
    "luxury_amenity_level",
    "city_neighborhood",
    "full_property_type",
    "month_number",
    "month_name",
    "season"
]

features_df = df[feature_columns].copy()

In [54]:
features_df.head()

,building_age,age_group,area_group,amenity_count,amenity_level,luxury_amenity_count,luxury_amenity_level,city_neighborhood,full_property_type,month_number,month_name,season
0,<NA>,NaN,بسیار بزرگ,0,کم,0,کم,karaj|mehrshahr,temporary-rent|villa,8,August,تابستان
1,20,میان سال,متوسط,3,متوسط,0,کم,tehran|gholhak,residential-sell|apartment-sell,5,May,بهار
2,3,نوساز,خیلی بزرگ,3,متوسط,0,کم,tehran|tohid,residential-rent|apartment-rent,10,October,پاییز
3,4,نوساز,بزرگ,2,کم,0,کم,tehran|elahiyeh,commercial-rent|office-rent,6,June,تابستان
4,1,نوساز,بزرگ,4,متوسط,0,کم,mashhad|emamreza,residential-sell|apartment-sell,5,May,بهار


In [19]:
features_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 999953 entries, 0 to 999999
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   building_age          815823 non-null  Int16   
 1   age_group             815823 non-null  category
 2   area_group            980387 non-null  category
 3   amenity_count         999953 non-null  Int64   
 4   amenity_level         999953 non-null  category
 5   luxury_amenity_count  999953 non-null  Int64   
 6   luxury_amenity_level  999953 non-null  category
 7   city_neighborhood     999953 non-null  object  
 8   full_property_type    999953 non-null  object  
 9   month_number          999953 non-null  int32   
 10  month_name            999953 non-null  object  
 11  season                999953 non-null  object  
dtypes: Int16(1), Int64(2), category(4), int32(1), object(4)
memory usage: 65.8+ MB


In [56]:
features_df.isna().sum()

building_age            184172
age_group               184172
area_group               19606
amenity_count                0
amenity_level                0
luxury_amenity_count         0
luxury_amenity_level         0
city_neighborhood            0
full_property_type           0
month_number                 0
month_name                   0
season                       0
dtype: int64

In [23]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'rent_mode',
       'rent_value', 'rent_to_single', 'rent_type', 'price_mode',
       'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform',
       'transformable_price', 'transformable_credit', 'transformed_credit',
       'transformable_rent', 'transformed_rent', 'land_size', 'building_size',
       'deed_type', 'has_business_deed', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas',
       'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'regular_person_capacity', 'extra_person

In [ ]:
# df.drop(columns=feature_columns, inplace=True)

BooleanDtype

In [58]:
df.shape

(1000000, 63)

In [59]:
features_df.shape

(1000000, 12)

In [20]:
df.to_feather("../Outputs/28_df.feather")



In [ ]:
# df.to_csv("../Outputs/final_df.csv", index=False, encoding="utf-8-sig")